# Object Detection on PASCAL VOC 2012
### End-to-end ML project · readable and configurable

**Goal:** locate and classify everyday objects in new images. Output: boxes, labels and confidence scores for 20 VOC classes.

**Workflow:** setup → **data cleaning** → **preprocessing** → baseline → **fine-tuning** → **hyperparameter tuning** → error analysis → optimization → final evaluation → demo export.

The notebook orchestrates short functions in `src/`. Open those modules to change implementation details. Do not submit invented results: the tables and report fields are filled by actual runs. No trained weights or VOC data are bundled.

**Recommended:** Python 3.11/3.12 and a free Colab/Kaggle GPU. CPU supports testing and inference; full training can be slow. Free GPU availability and runtime limits vary. See README for archive upload and persistence instructions.

## 0. Setup
Run from the extracted `voc_detection` directory. In Colab, upload `voc_detection_project.zip`, unzip it, and `%cd /content/voc_detection` first. Enable a GPU in Runtime settings.

Install the matching PyTorch/Torchvision pair using the command in README for your hardware. On Colab, first check the preinstalled pair below; avoid replacing a working CUDA build with CPU wheels.

In [ ]:
# Uncomment only if dependencies are missing, then restart the kernel.
# %pip install -r requirements.txt
from pathlib import Path
assert Path('src').is_dir(), 'Change the working directory to the extracted voc_detection folder.'
import torch, torchvision
print('PyTorch:', torch.__version__, 'Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
import json
import shutil
import gc
from dataclasses import replace
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from src.config import Config, CLASSES, seed_everything, write_json
from src.data import download_voc, audit_voc, make_splits, class_counts, loader, VOCDataset
from src.model import build_model, save_checkpoint, load_checkpoint
from src.engine import train_experiment, predict_loader, benchmark
from src.metrics import evaluate_detections, error_analysis
from src.visualize import draw_detections

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# A fresh directory prevents accidental overwriting or mixing experiment results.
RUN_NAME = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%f')
RUN_DIR = Path('artifacts') / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
DATA_ROOT = Path('data')
QUICK_RUN = False  # First verify the workflow; set False for submission experiments.
CFG = Config()
if QUICK_RUN:
    CFG = replace(CFG, epochs=1, train_limit=96, tune_limit=48, eval_limit=48,
                  min_size=320, max_size=640)
seed_everything(CFG.seed)
write_json(RUN_DIR / 'environment.json', {
    'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
    'device': str(DEVICE), 'quick_run': QUICK_RUN, 'config': CFG.dictionary()})
print('Device:', DEVICE, '| Run:', RUN_DIR, '| Quick run:', QUICK_RUN)

## 1. Data cleaning
Download the official VOC 2012 train/validation archive (several GB; reserve roughly 8–10 GB of disk space for archive and extracted files). Training uses only official `train`; official `val` is held out for final evaluation. VOC test annotations are not provided here, so this is **held-out validation performance**, not an official test-server result.

We decode images, check XML, verify class names and finite coordinates, clip boxes to image bounds, discard invalid boxes, and log every change. Images with no valid boxes remain background examples. We remove exact decoded-pixel duplicates, keeping official validation images first. Near-duplicates and possible overlap with pretrained COCO data are not audited.

VOC XML boxes use 1-based inclusive coordinates; the model uses 0-based half-open `XYXY`. We subtract one only from `xmin/ymin`. Training includes valid difficult objects; evaluation ignores difficult instances according to VOC matching rules.

In [ ]:
# If you already have the dataset, set VOC_DIR to .../VOCdevkit/VOC2012 instead.
VOC_DIR = download_voc(DATA_ROOT)
clean, cleaning_events = audit_voc(VOC_DIR, RUN_DIR)
print({name: len(records) for name, records in clean.items()})
print('Logged cleaning events:', len(cleaning_events))
display(pd.DataFrame(cleaning_events).head(15))

## 2. Preprocessing and exploratory analysis
Split official training images into **fit (80%)** and **tune (20%)**, using a fixed seed. Exact duplicates are removed before this split. Tune selects hyperparameters, checkpoints and deployment resolution; final evaluation is untouched until section 8.

The split is random, not multilabel-stratified. Check class coverage below, especially in quick mode. Classes without non-difficult ground truth have undefined AP and are excluded from that split's mean; compare models on identical records.

Preprocessing converts RGB pixels to float tensors in `[0,1]`. The detector internally resizes and normalizes inputs; do not normalize a second time. The only training augmentation is horizontal flip, applied to both image and boxes. Evaluation has no random augmentation.

In [ ]:
parts = make_splits(clean, CFG, RUN_DIR)
counts = pd.DataFrame({key: class_counts(records) for key, records in parts.items()})
display(counts)
print('Image counts:', {k: len(v) for k, v in parts.items()})
if (counts == 0).any().any():
    print('WARNING: Some classes lack ground truth. Use the full split for meaningful reporting.')
counts.plot.bar(figsize=(13, 4), title='Non-difficult object counts by class')
plt.tight_layout(); plt.show()
widths = [r['width'] for r in parts['fit']]
heights = [r['height'] for r in parts['fit']]
plt.scatter(widths, heights, s=8, alpha=.35)
plt.xlabel('Image width'); plt.ylabel('Image height'); plt.title('Training image sizes'); plt.show()

In [ ]:
# Ground truth visualization: confirm alignment before training.
sample = parts['fit'][0]
with Image.open(sample['path']) as im:
    truth = {'boxes': [o['box'] for o in sample['objects']],
             'labels': [o['label'] for o in sample['objects']],
             'scores': [1.0] * len(sample['objects'])}
    display(draw_detections(im, truth))
image_tensor, sample_target = VOCDataset([sample], flip_probability=1.0)[0]
flipped_image = Image.fromarray((image_tensor.permute(1,2,0).numpy()*255).astype('uint8'))
display(draw_detections(flipped_image, {
    **sample_target, 'scores': torch.ones(len(sample_target['boxes']))}))

## 3. Model and pretrained baseline
**Faster R-CNN + MobileNetV3-Large + Feature Pyramid Network:** the backbone extracts multiscale features; a region proposal network proposes boxes; ROI heads predict classes and refine box coordinates; NMS removes redundant boxes.

Use pretrained COCO weights to reduce data and compute needs. We copy the 20 corresponding classifier/regressor rows into a VOC-specific head, including COCO/VOC name aliases. This provides a meaningful zero-VOC-training baseline. Fine-tuning then adapts the detector to VOC.

**Limitation:** VOC classes overlap with COCO classes, and image overlap across pretraining corpora has not been audited. Baseline scores are not evidence of learning from scratch. The submission must disclose this initialization.

The backbone can be frozen for a cheaper adaptation experiment. Frozen batch normalization is kept consistent when saving/reloading checkpoints.

In [ ]:
seed_everything(CFG.seed)
model = build_model(CFG).to(DEVICE)
baseline_predictions, baseline_targets = predict_loader(model, loader(parts['tune'], CFG), DEVICE)
baseline_metrics = evaluate_detections(baseline_predictions, baseline_targets)
write_json(RUN_DIR / 'baseline_tune.json', baseline_metrics)
print('Pretrained baseline tune mAP50:', baseline_metrics['map50'])
del model
gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

## 4. Fine-tuning
Optimize the sum of classifier, box-regression, RPN objectness and RPN box-regression losses with SGD. Use CUDA mixed precision when available, gradient clipping, a step learning-rate schedule and early stopping based on **tune mAP50**. Save the best checkpoint, full loss history and trial configuration.

The checkpoint is for inference, not an exact mid-epoch optimizer resume. If a free session disconnects, completed runs remain reusable if you saved them externally; an interrupted trial must restart. Save artifacts outside ephemeral notebook storage.

In [ ]:
baseline_cfg = replace(CFG, freeze_backbone=True)
frozen_result = train_experiment(baseline_cfg, parts['fit'], parts['tune'],
                                 RUN_DIR / 'frozen_backbone', DEVICE)

## 5. Hyperparameter tuning and controlled experiments
Use a small explicit search rather than an expensive blind sweep. The default trial changes both learning rate and backbone training relative to the frozen baseline, so **do not attribute that comparison to one factor**. The two full-backbone trials isolate learning rate. The no-flip trial isolates augmentation at the same learning rate.

Every trial uses the same seed, splits, resolution, epoch budget and pretrained initialization. Select by tune mAP50 only. Quick mode tests plumbing with tiny subsets and one epoch; it does not establish a reliable ranking. For stronger evidence, run full data and repeat the winning settings with several seeds.

In [ ]:
trial_configs = {
    'full_lr_005': replace(CFG, lr=.005, freeze_backbone=False),
    'full_lr_001': replace(CFG, lr=.001, freeze_backbone=False),
    'full_no_flip': replace(CFG, lr=.005, freeze_backbone=False, flip_probability=0.),
}
results = [frozen_result]
for name, trial_cfg in trial_configs.items():
    results.append(train_experiment(trial_cfg, parts['fit'], parts['tune'], RUN_DIR / name, DEVICE))
experiments = pd.DataFrame(results).sort_values('best_tune_map50', ascending=False)
experiments.to_csv(RUN_DIR / 'experiments.csv', index=False)
display(experiments[['run','lr','freeze_backbone','flip_probability',
                     'best_tune_map50','epochs_completed','seconds']])
best = experiments.iloc[0]
BEST_CHECKPOINT = Path(best['checkpoint'])
model, best_cfg, checkpoint_metadata = load_checkpoint(BEST_CHECKPOINT, DEVICE)
print('Selected checkpoint:', BEST_CHECKPOINT)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for result in results:
    history = pd.read_json(RUN_DIR / result['run'] / 'history.json')
    axes[0].plot(history['epoch'], history['train_loss'], marker='o', label=result['run'])
    axes[1].plot(history['epoch'], history['tune_map50'], marker='o', label=result['run'])
axes[0].set(title='Training loss', xlabel='Epoch', ylabel='Loss')
axes[1].set(title='Tuning mAP50', xlabel='Epoch', ylabel='mAP50', ylim=(0,1))
axes[1].legend(); plt.tight_layout()
plt.savefig(RUN_DIR / 'training_curves.png', dpi=150); plt.show()

## 6. Evaluation methodology and error analysis
**Primary metric:** VOC 2012-style continuous interpolated mAP at IoU > 0.50, averaged over classes with non-difficult ground truth. Predictions are sorted by confidence; each non-difficult object matches at most once; duplicate detections are false positives. Matches to difficult instances are ignored.

**Secondary metric:** `map50_95_voc` averages this VOC-style AP over IoU 0.50:0.05:0.95. It is **not official COCO mAP**, which uses different recall sampling and evaluation conventions. Max detections per image is 100; evaluation score floor is 0.001. The app's confidence slider does not change the AP calculation.

Analyze tuning errors now. Diagnostics at confidence 0.5 distinguish missed objects, duplicates, class mistakes, localization mistakes and background detections. These categories are a greedy heuristic, not a formal causal error decomposition.

In [ ]:
tune_predictions, tune_targets = predict_loader(model, loader(parts['tune'], best_cfg), DEVICE)
tune_metrics = evaluate_detections(tune_predictions, tune_targets)
write_json(RUN_DIR / 'selected_tune_metrics.json', tune_metrics)
errors = pd.DataFrame(error_analysis(tune_predictions, tune_targets))
errors['error_count'] = errors[['duplicate','wrong_class','localization','background','missed']].sum(axis=1)
errors.to_csv(RUN_DIR / 'tune_errors.csv', index=False)
display(errors.drop(columns=['image_index']).sum().to_frame('count'))
pd.Series(tune_metrics['per_class_ap50']).plot.bar(figsize=(12, 4), ylim=(0,1), title='Tuning AP50 by class')
plt.tight_layout(); plt.savefig(RUN_DIR / 'tune_ap50.png', dpi=150); plt.show()
for i in errors.nlargest(3, 'error_count')['image_index']:
    record = parts['tune'][int(i)]
    with Image.open(record['path']) as im:
        print(record['id'], errors.loc[errors['image_index'] == i].to_dict('records'))
        display(draw_detections(im, tune_predictions[int(i)]))

## 7. Optimization: inference resolution versus latency
Try a smaller input resolution on the **same selected checkpoint**. We measure quality and synchronized batch-one model latency on tuning images. Timing includes tensor transfer, internal resize, detector and NMS, but excludes file decoding, UI rendering and network overhead. Preload at most 10 representative images and report warm-up, median, p95, device and thread count.

Choose the fastest policy only if it stays within an absolute **0.02 mAP50** drop on tuning data. This is a predefined deployment constraint, not a guaranteed speedup. Resolution changes compute, not parameter count or checkpoint size. If no faster policy qualifies, keep the original. GPU peak allocation is reported when CUDA is used; CPU memory is not measured. Both policies use FP32 inference for a fair resolution comparison.

In [ ]:
policies = [
    {'name': 'original', 'min_size': best_cfg.min_size, 'max_size': best_cfg.max_size},
    {'name': 'smaller', 'min_size': max(160, best_cfg.min_size * 2 // 3),
     'max_size': max(320, best_cfg.max_size * 2 // 3)},
]
MAX_MAP50_DROP = .02
optimization_rows = []
for policy in policies:
    model.transform.min_size = (policy['min_size'],)
    model.transform.max_size = policy['max_size']
    pred, gt = predict_loader(model, loader(parts['tune'], best_cfg), DEVICE)
    quality = evaluate_detections(pred, gt)
    timing = benchmark(model, VOCDataset(parts['tune']), DEVICE,
                       warmup=3, repeats=10 if QUICK_RUN else 30)
    optimization_rows.append({**policy, 'tune_map50': quality['map50'],
                               'tune_map50_95_voc': quality['map50_95_voc'],
                               'checkpoint_mb': BEST_CHECKPOINT.stat().st_size / 2**20,
                               'parameters': sum(p.numel() for p in model.parameters()), **timing})
optimization = pd.DataFrame(optimization_rows)
reference_ap = optimization_rows[0]['tune_map50']
assert reference_ap is not None, 'Need evaluable classes in tuning data.'
eligible = optimization[optimization['tune_map50'] >= reference_ap - MAX_MAP50_DROP]
selected = eligible.sort_values('median_ms', kind='stable').iloc[0]
SELECTED_POLICY = next(p for p in policies if p['name'] == selected['name'])
optimization.to_csv(RUN_DIR / 'optimization_tune.csv', index=False)
write_json(RUN_DIR / 'selected_policy.json', SELECTED_POLICY)
display(optimization)
print('Selected policy:', SELECTED_POLICY)

## 8. Final held-out evaluation
The model and deployment policy are now fixed. Evaluate original and selected deployment resolutions on official validation images (or the clearly labeled quick subset). Report both as a final comparison; do not change the chosen policy based on these scores. Repeated development against this set would make it a tuning set.

A full run excludes images rejected by cleaning; compare to external VOC results cautiously. `splits.json` records exactly which IDs were evaluated.

In [ ]:
final_rows = []
final_predictions, final_targets = None, None
for policy in policies:
    model.transform.min_size = (policy['min_size'],)
    model.transform.max_size = policy['max_size']
    predictions, targets = predict_loader(model, loader(parts['evaluation'], best_cfg), DEVICE)
    metrics = evaluate_detections(predictions, targets)
    write_json(RUN_DIR / f"final_{policy['name']}_metrics.json", metrics)
    final_rows.append({**policy, 'map50': metrics['map50'],
                       'map50_95_voc': metrics['map50_95_voc'],
                       'images': metrics['images'], 'classes_with_gt': metrics['classes_with_gt']})
    if policy['name'] == SELECTED_POLICY['name']:
        final_predictions, final_targets = predictions, targets
final_table = pd.DataFrame(final_rows)
final_table.to_csv(RUN_DIR / 'final_comparison.csv', index=False)
display(final_table)
final_errors = pd.DataFrame(error_analysis(final_predictions, final_targets))
final_errors.to_csv(RUN_DIR / 'final_errors.csv', index=False)
for i in range(min(3, len(parts['evaluation']))):
    record = parts['evaluation'][i]
    with Image.open(record['path']) as im:
        annotated = draw_detections(im, final_predictions[i])
        annotated.save(RUN_DIR / f"prediction_{record['id']}.jpg")
        display(annotated)

## 9. Export and use on new images
Export a portable state-dict checkpoint plus a deployment policy for the local Streamlit app. Reloading does not fetch weights. It uses the same class mapping, preprocessing and selected resolution. Copy/download the entire `artifacts/` folder before a free GPU session ends. The app defaults to CPU inference and runs locally without a paid service.

The best checkpoint is selected using the tuning split and is not retrained on tuning data; this keeps the selected checkpoint directly comparable to the measured metrics.

In [ ]:
Path('artifacts').mkdir(exist_ok=True)
shutil.copy2(BEST_CHECKPOINT, 'artifacts/best.pt')
write_json('artifacts/deployment.json', {
    **SELECTED_POLICY, 'source_run': str(RUN_DIR),
    'quick_run': QUICK_RUN, 'classes': list(CLASSES),
    'checkpoint': 'best.pt', 'quality_budget_absolute_map50_drop': MAX_MAP50_DROP})
# Check exported artifacts work without pretrained downloads.
deployed_model, _, _ = load_checkpoint('artifacts/best.pt', DEVICE)
deployed_model.transform.min_size = (SELECTED_POLICY['min_size'],)
deployed_model.transform.max_size = SELECTED_POLICY['max_size']
print('Exported artifacts/best.pt and artifacts/deployment.json')
print('From a terminal in this folder: streamlit run app.py')

In [ ]:
# Optional: run on your own image, using precisely the app's preprocessing.
NEW_IMAGE_PATH = None  # e.g. 'my_photo.jpg'
if NEW_IMAGE_PATH:
    from PIL import ImageOps
    from torchvision.transforms.functional import pil_to_tensor
    image = ImageOps.exif_transpose(Image.open(NEW_IMAGE_PATH)).convert('RGB')
    with torch.inference_mode():
        prediction = deployed_model([pil_to_tensor(image).float().to(DEVICE)/255.])[0]
    display(draw_detections(image, prediction, .5))

## 10. Technical report and submission checklist
The next cell writes a factual metrics summary from your run. Complete `reports/TECHNICAL_REPORT.md` with your reasoning and actual observations; use saved CSV/JSON tables and prediction examples as evidence. Complete `AI_USAGE.md` to reflect what you actually used and checked.

Before submission: run full data, review coverage and learning curves, document hardware/limitations, include a trained checkpoint or reproducible training instructions, launch the app on a new image, and verify the repository in a clean environment. No paid hosting is required.

Be ready to explain coordinate conversion, losses, matching rules, why tuning uses a separate split, frozen versus trainable backbone, and the measured speed/quality tradeoff.

In [ ]:
summary = {
    'run_directory': str(RUN_DIR), 'quick_run': QUICK_RUN,
    'split_sizes': {k: len(v) for k,v in parts.items()},
    'pretrained_baseline_tune_map50': baseline_metrics['map50'],
    'winner': str(best['run']), 'winner_tune_map50': float(best['best_tune_map50']),
    'selected_policy': SELECTED_POLICY,
    'final_comparison': final_table.to_dict(orient='records'),
    'status': 'Smoke run only; run full data before submission.' if QUICK_RUN else 'Measured full-split run; review technical report.'
}
write_json(RUN_DIR / 'results_summary.json', summary)
print(json.dumps(summary, indent=2))
# Optional Colab download (includes model, metrics, audit and split manifests):
# shutil.make_archive('trained_artifacts', 'zip', 'artifacts')
# from google.colab import files
# files.download('trained_artifacts.zip')

## References
- [Torchvision Faster R-CNN MobileNetV3 documentation](https://docs.pytorch.org/vision/0.23/models/generated/torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn.html)
- [Torchvision detection fine-tuning tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html)
- [VOC 2012 development kit and evaluation rules](https://www.robots.ox.ac.uk/~vgg/projects/pascal/VOC/voc2012/htmldoc/index.html)
- [Matching PyTorch/Torchvision installation pairs](https://pytorch.org/get-started/previous-versions/)